# AI Powered GitHub Pull Request Risk Intelligence and Agentic Review Platform

## Machine Learning, LLM, RAG and Agentic RAG

### Project Objective

This project develops a locally hosted AI system for analysing GitHub
pull requests and identifying engineering risks, review bottlenecks,
policy conflicts, testing gaps and potential merge delays.

The solution combines:

- GitHub REST API data ingestion
- Data quality validation
- Exploratory data analysis
- Machine learning based merge delay prediction
- Rule based engineering risk assessment
- Local LLM analysis using Ollama
- Retrieval Augmented Generation
- Agentic RAG with controlled tool selection
- Static code analysis
- Explainable hybrid risk scoring
- ML, retrieval, LLM and agent evaluation
- Executive engineering analytics

### Main Business Questions

1. Which pull requests carry the highest engineering risk?
2. Which pull requests are likely to experience merge delays?
3. Which changes conflict with repository policies?
4. Which pull requests contain testing gaps?
5. Which deterministic and retrieved evidence supports each finding?
6. How reliable and grounded are the AI-generated findings?

## Solution Architecture

```text
Historical GitHub Pull Requests
              |
              v
     Data Ingestion and Caching
              |
              v
     Data Validation and Cleaning
              |
              v
      Feature Engineering
              |
              v
   Merge-Delay Prediction Model
              |
              +----------------------+
                                     |
Live GitHub Pull Request             |
              |                      |
              v                      |
       Rule-Based Risk               |
              |                      |
       +------+------+               |
       |             |               |
       v             v               v
   Local LLM        RAG       ML Probability
       |             |
       +------+------+
              |
       RAG Agent Tools
              |
       Static Analysis
              |
       Hybrid Risk Score
              |
   Evidence-Grounded PR Report
              |
   Monitoring and Executive Dashboard
```

### Proof of Concept Scope

All components are implemented inside one Jupyter Notebook for local
development and demonstration.

In a production deployment, ingestion, model training, live inference,
vector retrieval, agent execution, monitoring and user interfaces would
normally be separated into independently deployable services.

In [3]:
%pip install -q --upgrade pip
%pip install -q requests pandas numpy scipy scikit-learn matplotlib
%pip install -q python-dotenv pydantic rich tqdm joblib ollama

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -q requests pandas numpy scipy scikit-learn matplotlib python-dotenv pydantic rich tqdm joblib ollama

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nProject folder:")
print(Path.cwd())

Python executable:
c:\Users\sabih\anaconda3\envs\github_pr_ai\python.exe

Python version:
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]

Project folder:
c:\Users\sabih\OneDrive\Desktop\Courseworks\AI GitHub PR Intelligence


In [2]:
import os
import sys
import json
import time
import random
import logging
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import sklearn
import ollama

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

print("All initial packages imported successfully.")

All initial packages imported successfully.


In [3]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
CACHE_DIR = DATA_DIR / "cache"
KNOWLEDGE_BASE_DIR = DATA_DIR / "knowledge_base"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
EVALUATION_DIR = DATA_DIR / "evaluation"

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
LOGS_DIR = PROJECT_ROOT / "logs"

PROJECT_DIRECTORIES = [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    CACHE_DIR,
    KNOWLEDGE_BASE_DIR,
    VECTOR_STORE_DIR,
    EVALUATION_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    OUTPUTS_DIR,
    LOGS_DIR,
]

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print("Created project folders:\n")

for directory in PROJECT_DIRECTORIES:
    print("✓", directory.relative_to(PROJECT_ROOT))

Created project folders:

✓ data
✓ data\raw
✓ data\processed
✓ data\cache
✓ data\knowledge_base
✓ data\vector_store
✓ data\evaluation
✓ models
✓ reports
✓ outputs
✓ logs


In [4]:
LOG_FILE_PATH = LOGS_DIR / "pr_intelligence.log"

logger = logging.getLogger("github_pr_intelligence")
logger.setLevel(logging.INFO)
logger.handlers.clear()

log_formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler = logging.FileHandler(
    LOG_FILE_PATH,
    encoding="utf-8"
)
file_handler.setFormatter(log_formatter)

console_handler = logging.StreamHandler()
console_handler.setFormatter(log_formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("Logging system initialised successfully.")

print("\nLog file:")
print(LOG_FILE_PATH)

2026-07-29 03:41:58,098 | INFO | Logging system initialised successfully.



Log file:
c:\Users\sabih\OneDrive\Desktop\Courseworks\AI GitHub PR Intelligence\logs\pr_intelligence.log
